<a href="https://colab.research.google.com/github/soumyasinghh/TrafficRiskDetection/blob/main/DeepLearningAssign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 1 : Implement Backpropagation in a Feed forward Neural Network

In [1]:
import numpy as np

# Activations
sig  = lambda z: 1/(1+np.exp(-z))
tanh = lambda z: np.tanh(z)
relu = lambda z: np.maximum(0,z)
dsig  = lambda z: sig(z)*(1-sig(z))
dtanh = lambda z: 1-np.tanh(z)**2
drelu = lambda z: (z>0).astype(float)

class NN:
    def __init__(self, layers, lr=0.1):
        self.lr = lr
        self.W = [np.random.randn(b,a)*np.sqrt(2/a) for a,b in zip(layers,layers[1:])]
        self.b = [np.zeros((n,1)) for n in layers[1:]]

    def forward(self, X):
        A, Z = [X], []
        for i,(w,b) in enumerate(zip(self.W,self.b)):
            z = w@A[-1]+b;  Z.append(z)
            A.append(sig(z) if i==len(self.W)-1 else tanh(z))
        return A, Z

    def backward(self, A, Z, y):
        m = y.shape[1]
        d = (A[-1]-y)/m
        dW, db = [], []
        for l in range(len(self.W)-1,-1,-1):
            dW.insert(0, d@A[l].T);  db.insert(0, d.sum(1,keepdims=True))
            if l: d = (self.W[l].T@d)*dtanh(Z[l-1])
        return dW, db

    def fit(self, X, y, epochs=3000, lr=None):
        if lr: self.lr=lr
        for ep in range(1,epochs+1):
            A,Z = self.forward(X)
            dW,db = self.backward(A,Z,y)
            for l in range(len(self.W)):
                self.W[l]-=self.lr*dW[l];  self.b[l]-=self.lr*db[l]
            if ep%500==0:
                loss = -np.mean(y*np.log(A[-1]+1e-9)+(1-y)*np.log(1-A[-1]+1e-9))
                acc  = np.mean((A[-1]>=.5)==y)*100
                print(f"Ep {ep:>4} | loss {loss:.4f} | acc {acc:.1f}%")

    def predict(self, X): return (self.forward(X)[0][-1]>=.5).astype(int)


# XOR demo
np.random.seed(42)
X = np.array([[0,0,1,1],[0,1,0,1]], dtype=float)
y = np.array([[0,1,1,0]], dtype=float)

net = NN([2,4,4,1])
net.fit(X, y, epochs=3000, lr=0.5)

print("\nXOR Predictions:")
probs = net.forward(X)[0][-1]
for i in range(4):
    print(f"  {int(X[0,i])} XOR {int(X[1,i])} -> {probs[0,i]:.4f}  {'OK' if net.predict(X)[0,i]==y[0,i] else 'FAIL'}")

Ep  500 | loss 0.0021 | acc 100.0%
Ep 1000 | loss 0.0009 | acc 100.0%
Ep 1500 | loss 0.0006 | acc 100.0%
Ep 2000 | loss 0.0004 | acc 100.0%
Ep 2500 | loss 0.0003 | acc 100.0%
Ep 3000 | loss 0.0003 | acc 100.0%

XOR Predictions:
  0 XOR 0 -> 0.0002  OK
  0 XOR 1 -> 0.9997  OK
  1 XOR 0 -> 0.9998  OK
  1 XOR 1 -> 0.0004  OK


What it means:

Training log — every 500 epochs it prints the loss and accuracy. Loss drops from 0.0021 → 0.0003, and accuracy hits 100% by epoch 500 and stays there.
XOR Predictions — the network correctly learned XOR. The output is a sigmoid probability (0–1):

0 XOR 0 = 0 → predicts 0.0002 (≈ 0) ✅
0 XOR 1 = 1 → predicts 0.9997 (≈ 1) ✅
1 XOR 0 = 1 → predicts 0.9998 (≈ 1) ✅
1 XOR 1 = 0 → predicts 0.0004 (≈ 0) ✅



All four cases classified correctly with very high confidence.

Assignment 2 : Implement the Convolutional Neural Network for 2 class classification task.

In [3]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# model
def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.MaxPool2d(2))

class CNN(nn.Module):
    def __init__(self, in_ch=3, img_size=64):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(in_ch, 32), conv_block(32, 64), conv_block(64, 128),
            nn.Flatten(),
            nn.Linear(128 * (img_size // 8) ** 2, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1)
        )
    def forward(self, x): return self.net(x)

# Data: CIFAR-10 cats(3) vs dogs(5)
def get_loaders(img_size=64, batch_size=32):
    tf = transforms.Compose([transforms.Resize((img_size, img_size)),
                              transforms.ToTensor(),
                              transforms.Normalize([0.5]*3, [0.5]*3)])

    def binary_subset(train):
        ds  = datasets.CIFAR10(root="./data", train=train, download=True, transform=tf)
        idx = [i for i, (_, y) in enumerate(ds) if y in (3, 5)]   # cats & dogs only

        class BinaryDataset(torch.utils.data.Dataset):
            def __init__(self, ds, idx): self.ds, self.idx = ds, idx
            def __len__(self): return len(self.idx)
            def __getitem__(self, i):
                img, lbl = self.ds[self.idx[i]]
                return img, 0 if lbl == 3 else 1   # cat=0, dog=1

        return BinaryDataset(ds, idx)

    train_ds = binary_subset(train=True)
    val_ds   = binary_subset(train=False)
    print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
    return DataLoader(train_ds, batch_size, shuffle=True), DataLoader(val_ds, batch_size)

# train / evaluation
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    loss_sum = correct = total = 0
    with torch.set_grad_enabled(train):
        for X, y in loader:
            X, y = X.to(device), y.float().unsqueeze(1).to(device)
            out  = model(X); loss = criterion(out, y)
            if train: optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(X)
            correct  += ((torch.sigmoid(out) >= 0.5).long() == y.long()).sum().item()
            total    += len(X)
    return loss_sum / total, correct / total

def train(epochs=20, lr=1e-3, img_size=64, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    train_loader, val_loader = get_loaders(img_size, batch_size)
    model     = CNN(img_size=img_size).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}")
    print("─" * 60)
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, optimizer, device, train=False)
        print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>9.4f} | {vl_acc:>8.4f}")

    torch.save(model.state_dict(), "cnn_binary.pth")
    print("\nModel saved → cnn_binary.pth")
    return model

model = train()

Device: cpu


100%|██████████| 170M/170M [00:02<00:00, 62.3MB/s]


Train samples: 10000 | Val samples: 2000

 Epoch | Train Loss | Train Acc |  Val Loss |  Val Acc
────────────────────────────────────────────────────────────
     1 |     0.7166 |    0.5792 |    0.6388 |   0.6485
     2 |     0.6373 |    0.6402 |    0.5977 |   0.6865
     3 |     0.5967 |    0.6770 |    0.5752 |   0.7205
     4 |     0.5657 |    0.7022 |    0.5140 |   0.7450
     5 |     0.5370 |    0.7207 |    0.4943 |   0.7610
     6 |     0.5178 |    0.7354 |    0.4865 |   0.7695
     7 |     0.4914 |    0.7456 |    0.4758 |   0.7660
     8 |     0.4788 |    0.7504 |    0.4875 |   0.7655
     9 |     0.4600 |    0.7735 |    0.4578 |   0.7760
    10 |     0.4424 |    0.7772 |    0.4685 |   0.7855
    11 |     0.4294 |    0.7870 |    0.4557 |   0.7820
    12 |     0.4136 |    0.7997 |    0.4459 |   0.7950
    13 |     0.3971 |    0.8071 |    0.4464 |   0.7880
    14 |     0.3773 |    0.8188 |    0.4517 |   0.7990
    15 |     0.3744 |    0.8172 |    0.4467 |   0.7940
    16 |     0.35